# Data Cleaning Process

I use 16,495 U.S. job postings from 307 S&P 1,500 firms. The job-posting data come from Techmap and were collected between March 1 and March 9, 2022. Firm-level financial data come from Wharton Research Data Services (WRDS) and provide ROA in 2020 and 2021.

Before creating embeddings, I remove firm names and common aliases, URLs, and email addresses to reduce firm-identity leakage. I then split the data into train, validation, and holdout sets at the firm level so that postings from the same firm are not shared across splits.

I considered removing job titles as a robustness check (labeled as an option in the first cell). However, occupation remains inferable from responsibilities, skills, certifications, and industry-specific language, so removing the title does not eliminate occupational composition.

In [2]:
import pandas as pd
import re

# Load data
df = pd.read_csv("/Users/jiaoweigong/Desktop/HypotheSAEs/demo_data/JobPosting_2022Mar1-9.csv")

# Common firm aliases to remove from posting text.
# I used Codex to help generate/check likely aliases based on company names.
FIRM_ALIASES = {
    "advance auto parts": ["Advance Auto Parts"],
    "tjx": ["The TJX Companies", "TJX Companies", "TJX"],
    "bank of america": ["Bank of America", "BofA"],
    "marten transport": ["Marten Transport"],
    "stepan": ["Stepan Company", "Stepan Co", "Stepan"],
    "cracker barrel": ["Cracker Barrel Old Country Store", "Cracker Barrel"],
    "healthcare services group": ["Healthcare Services Group"],
    "pnc financial": ["The PNC Financial Services Group", "PNC Financial Services Group", "PNC Financial Services", "PNC"],
    "dick's sporting goods": ["DICK'S Sporting Goods", "Dicks Sporting Goods"],
    "coca-cola consolidated": ["Coca-Cola Consolidated", "Coca Cola Consolidated"],
    "marriott vacations worldwide": ["Marriott Vacations Worldwide"],
    "flowserve": ["Flowserve Corporation", "Flowserve"],
    "arrow electronics": ["Arrow Electronics"],
    "exelon": ["Exelon Corporation", "Exelon"],
    "hilton grand vacations": ["Hilton Grand Vacations"],
    "sherwin-williams": ["The Sherwin-Williams Company", "Sherwin-Williams Company", "Sherwin-Williams", "Sherwin Williams"],
    "option care health": ["Option Care Health"],
    "paychex": ["Paychex Inc", "Paychex"],
    "pg&e": ["PG&E Corporation", "PG&E", "Pacific Gas and Electric"],
    "avis budget": ["Avis Budget Group", "Avis Budget"],
    "united natural foods": ["United Natural Foods", "UNFI"],
    "eaton": ["Eaton Corporation", "Eaton"],
    "hca healthcare": ["HCA Healthcare", "HCA"],
    "chipotle": ["Chipotle Mexican Grill", "Chipotle"],
    "southern company": ["Southern Company", "The Southern Company"],
    "lowe": ["Lowe's Companies", "Lowes Companies", "Lowe's", "Lowes"],
    "colgate-palmolive": ["Colgate-Palmolive Company", "Colgate-Palmolive", "Colgate Palmolive"],
    "avalonbay": ["AvalonBay Communities", "AvalonBay"],
    "morgan stanley": ["Morgan Stanley"],
    "altec industries": ["Altec Industries", "Altec"],
    "aramark": ["Aramark"],
    "trane technologies": ["Trane Technologies", "Trane"],
    "jpmorgan chase": ["JPMorgan Chase & Co", "JPMorgan Chase", "J.P. Morgan", "JP Morgan", "JPMorgan"],
    "armstrong world industries": ["Armstrong World Industries"],
    "central garden": ["Central Garden & Pet", "Central Garden and Pet"],
    "rogers corporation": ["Rogers Corporation", "Rogers"],
    "ameriprise": ["Ameriprise Financial", "Ameriprise"],
    "hanesbrands": ["Hanesbrands Inc", "Hanesbrands"],
    "lennox international": ["Lennox International", "Lennox"],
    "american electric power": ["American Electric Power Company", "American Electric Power", "AEP"],
    "hni corporation": ["HNI Corporation", "HNI"],
    "bank of new york mellon": ["Bank of New York Mellon", "The Bank of New York Mellon", "BNY Mellon"],
    "public storage": ["Public Storage"],
    "kb home": ["KB Home"],
    "cerner": ["Cerner Corporation", "Cerner"],
    "live nation": ["Live Nation Entertainment", "Live Nation"],
    "cushman": ["Cushman & Wakefield", "Cushman and Wakefield"],
    "dxp enterprises": ["DXP Enterprises", "DXP"],
    "first american financial": ["First American Financial Corporation", "First American Financial"],
    "firstenergy": ["FirstEnergy Corp", "FirstEnergy"],
    "synchrony": ["Synchrony Financial", "Synchrony"],
    "ugi": ["UGI Corporation", "UGI"],
    "northrop grumman": ["Northrop Grumman"],
    "cnh industrial": ["CNH Industrial"],
    "dominion energy": ["Dominion Energy"],
    "advanced drainage systems": ["Advanced Drainage Systems"],
    "extra space storage": ["Extra Space Storage"],
    "fox corporation": ["FOX Corporation", "Fox Corporation", "FOX"],
    "johnson": ["Johnson & Johnson", "Johnson and Johnson", "J&J"],
    "hewlett packard enterprise": ["Hewlett Packard Enterprise", "HPE"],
    "r systems": ["R Systems"],
    "indotronix international": ["Indotronix International"],
    "aecom": ["AECOM"],
    "burlington stores": ["Burlington Stores", "Burlington"],
    "rollins": ["Rollins, Inc.", "Rollins Inc", "Rollins"],
    "molina healthcare": ["Molina Healthcare", "Molina"],
    "advanced energy industries": ["Advanced Energy Industries", "Advanced Energy"],
    "cavco industries": ["Cavco Industries", "Cavco"],
    "cardinal health": ["Cardinal Health"],
    "valley national bank": ["Valley National Bank", "Valley Bank"],
    "ford motor": ["Ford Motor Company", "Ford Motor", "Ford"],
    "jack in the box": ["Jack in the Box"],
    "marsh": ["Marsh & McLennan Companies", "Marsh and McLennan Companies", "Marsh McLennan", "Marsh"],
    "maximus": ["MAXIMUS, Inc.", "MAXIMUS Inc", "MAXIMUS"],
    "wec energy": ["WEC Energy Group", "WEC Energy", "WEC"],
    "data incorporated": ["Data Incorporated"],
    "adtalem": ["Adtalem Global Education", "Adtalem"],
    "cno financial": ["CNO Financial Group", "CNO Financial", "CNO"],
    "becton": ["Becton, Dickinson & Company", "Becton Dickinson & Company", "Becton Dickinson", "BD"],
    "lumen technologies": ["Lumen Technologies", "Lumen"],
    "pc connection": ["PC Connection, Inc.", "PC Connection Inc", "PC Connection"],
    "thermo fisher scientific": ["Thermo Fisher Scientific", "Thermo Fisher"],
    "ralph lauren": ["Ralph Lauren Corporation", "Ralph Lauren"],
    "diamondback energy": ["Diamondback Energy"],
    "azz": ["AZZ Inc.", "AZZ Inc", "AZZ"],
    "performance food group": ["Performance Food Group", "PFG"],
    "walt disney": ["The Walt Disney Company", "Walt Disney Company", "Walt Disney", "Disney"],
    "magna international": ["Magna International Inc.", "Magna International", "Magna"]
}

# Normalize firm names so matching is less sensitive to punctuation.
def normalize_name(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().replace("&", "and")
    x = re.sub(r"[^a-z0-9]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()

# Create a simplified version of orgname by removing "(The)" and common corporate suffixes.
def simplify_orgname(org):
    if pd.isna(org):
        return ""
    org = re.sub(r"\(\s*The\s*\)", " ", str(org), flags=re.IGNORECASE)
    org = re.sub(
        r"\b(Inc|Incorporated|Corp|Corporation|Company|Companies|Co|Ltd|Limited|LLC|PLC|Holdings)\.?\b",
        " ", org, flags=re.IGNORECASE
    )
    return re.sub(r"\s+", " ", org).strip()

# Clean each job posting.
# For every posting, remove:
#   1. the exact firm name recorded in orgname,
#   2. a simplified version of that firm name,
#   3. common aliases for larger firms from FIRM_ALIASES,
#   4. URLs and email addresses.
# Longer firm names are removed first to avoid leaving partial names behind.
def clean_text(row):
    text, org = row["text"], row["orgname"]
    if pd.isna(text):
        return ""

    text = str(text)
    names_to_remove = []

    if pd.notna(org):
        full_org = str(org).strip()
        simple_org = simplify_orgname(org)
        normalized_org = normalize_name(org)

        names_to_remove.append(full_org)
        if simple_org:
            names_to_remove.append(simple_org)

        # Match the current firm to the manually checked alias dictionary.
        for firm_key, aliases in FIRM_ALIASES.items():
            if normalize_name(firm_key) in normalized_org:
                names_to_remove.extend(aliases)

    # (Optional) robustness check: Remove the job title from the posting text.
    # Left commented out in the main analysis because occupation-specific information also appears throughout responsibilities, skills, and requirements.
    #
    # jobname = row["jobname"]
    # if pd.notna(jobname):
    #     names_to_remove.append(str(jobname).strip())

    # Remove longest names first.
    for name in sorted(set(names_to_remove), key=len, reverse=True):
        if name and len(name) >= 3:
            text = re.sub(re.escape(name), " ", text, flags=re.IGNORECASE)

     # Remove URLs and email addresses.
    text = re.sub(r"https?://\S+|www\.\S+", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", " ", text)

    # Collapse extra whitespace left by removed text.
    return re.sub(r"\s+", " ", text).strip()

df["text_clean"] = df.apply(clean_text, axis=1)

pd.set_option("display.max_colwidth", None)
display(df[["orgname", "jobname", "text", "text_clean"]].sample(5, random_state=123))

orgname  \
6497   Cracker Barrel Old Country Store   
15612             The TJX Companies Inc   
8664    Healthcare Services Group, Inc.   
5686                Centene Corporation   
8549    Healthcare Services Group, Inc.   

                                       jobname  \
6497                                      Host   
15612  Retail Associate Part-Time *Now Hiring*   
8664                               Housekeeper   
5686                 Licensed Social Worker-NJ   
8549                               Housekeeper   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [3]:
# Split data into train, validation, and holdout sets by firm. 
# Each firm is assigned to only one of the three sets.
from sklearn.model_selection import train_test_split

# One row per firm
firm_df = df[["orglabel", "decline"]].drop_duplicates().reset_index(drop=True)
assert df.groupby("orglabel")["decline"].nunique().max() == 1

# 70% train, 15% val, 15% holdout
trainval_firms, holdout_firms = train_test_split(
    firm_df, test_size=0.15, random_state=123, stratify=firm_df["decline"]
)

train_firms, val_firms = train_test_split(
    trainval_firms, test_size=0.15/0.85, random_state=123,
    stratify=trainval_firms["decline"]
)

train_df = df[df["orglabel"].isin(train_firms["orglabel"])].copy()
val_df = df[df["orglabel"].isin(val_firms["orglabel"])].copy()
holdout_df = df[df["orglabel"].isin(holdout_firms["orglabel"])].copy()

# HypotheSAEs inputs
texts = train_df["text_clean"].tolist()
labels = train_df["decline"].astype(int).values
val_texts = val_df["text_clean"].tolist()
holdout_texts = holdout_df["text_clean"].tolist()
holdout_labels = holdout_df["decline"].astype(int).values

# Check
print("Train:", train_df.shape, train_df["orglabel"].nunique(), "firms")
print("Val:", val_df.shape, val_df["orglabel"].nunique(), "firms")
print("Holdout:", holdout_df.shape, holdout_df["orglabel"].nunique(), "firms")

print("Overlap:",
      len(set(train_df["orglabel"]) & set(val_df["orglabel"])),
      len(set(train_df["orglabel"]) & set(holdout_df["orglabel"])),
      len(set(val_df["orglabel"]) & set(holdout_df["orglabel"])))

# Save train, validation, and holdout as JSONL
base_dir = "/Users/jiaoweigong/Desktop/HypotheSAEs/demo_data"

train_df.to_json(f"{base_dir}/JobPosting_train.json", orient="records", lines=True, force_ascii=False)
val_df.to_json(f"{base_dir}/JobPosting_val.json", orient="records", lines=True, force_ascii=False)
holdout_df.to_json(f"{base_dir}/JobPosting_holdout.json", orient="records", lines=True, force_ascii=False)

Train: (14274, 8) 214 firms
Val: (950, 8) 46 firms
Holdout: (1271, 8) 47 firms
Overlap: 0 0 0


# HypotheSAEs Quickstart

This notebook demonstrates basic usage of HypotheSAEs.
By default we use `gpt-5.2` for hypothesis generation (interpreting neurons) and `gpt-5-mini` for text annotation.

Set `OPENAI_KEY_SAE` below for OpenAI-hosted requests. If you are using a local OpenAI-compatible endpoint (for example, vLLM server mode), set `OPENAI_BASE_URL`; the key is optional for non-OpenAI hosts.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.environ['OPENAI_KEY_SAE'] = '...'  # delete my API key for security
# os.environ['OPENAI_BASE_URL'] = 'http://127.0.0.1:8000/v1'  # Optional: local OpenAI-compatible endpoint

import numpy as np
import pandas as pd

from hypothesaes.quickstart import train_sae, interpret_sae, generate_hypotheses, evaluate_hypotheses
from hypothesaes.embedding import get_openai_embeddings, get_local_embeddings

INTERPRETER_MODEL = "gpt-5.2"
ANNOTATOR_MODEL = "gpt-5-mini"

# Optional request kwargs forwarded directly to the Responses API.
# Keep empty by default; set only what you need at call time.
# Example: INTERPRET_LLM_KWARGS = {"reasoning_effort": "low"}
INTERPRET_LLM_KWARGS = {}
ANNOTATION_LLM_KWARGS = {}

N_WORKERS_ANNOTATION = 10  # Lower if hitting API rate limits


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


**Load cleaned train and validation data**

In [28]:
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    prefix = "../"
else:
    prefix = "./"

base_dir = os.path.join(prefix, "demo_data")
train_df = pd.read_json(os.path.join(base_dir, "JobPosting_train.json"), lines=True)
val_df = pd.read_json(os.path.join(base_dir, "JobPosting_val.json"), lines=True)

texts = train_df['text_clean'].tolist()
labels = train_df["decline"].astype(int).values
val_texts = val_df['text_clean'].tolist() # These are only used for early stopping of SAE training, so we don't need labels.

In [29]:
print(train_df.columns.tolist())
print(train_df.dtypes)

print(train_df.iloc[0])

['orgname', 'orglabel', 'jobname', 'text', 'roa2020', 'roa2021', 'decline', 'text_clean']
orgname           str
orglabel          str
jobname           str
text              str
roa2020       float64
roa2021       float64
decline         int64
text_clean        str
dtype: object
orgname                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

**Compute text embeddings for your dataset**

We'll compute text embeddings for a training set, and optionally a validation set. The validation embeddings are used for SAE eval and early-stopping during training.

Embeddings will be stored in the `emb_cache` directory (or `os.environ["EMB_CACHE_DIR"]` if you set it) using the `cache_name` parameter, so you only need to compute embeddings once.

For embeddings, you can use OpenAI (`get_openai_embeddings`) or a local sentence-transformers model (`get_local_embeddings`).

LLM inference (interpretation/annotation) uses the unified OpenAI-compatible API path. Point it to your local endpoint by setting `OPENAI_BASE_URL` when needed.


In [ ]:
import time
from openai import RateLimitError

EMBEDDER = "text-embedding-3-small" # OpenAI
# EMBEDDER = "nomic-ai/modernbert-embed-base" # Huggingface model, will run locally
CACHE_NAME = f"jobposting_{EMBEDDER}"


# I used Codex here to help optimize the embedding pipeline for API limits by processing texts in smaller chunks and retrying after rate-limit errors.
all_texts = texts + val_texts
text2embedding = {}

for i in range(0, len(all_texts), 500):
    chunk = all_texts[i:i+500]

    while True:
        try:
            result = get_openai_embeddings(
                chunk, model=EMBEDDER, batch_size=16, n_workers=1,
                cache_name=CACHE_NAME
            )
            break
        except RateLimitError:
            print("Rate limit reached. Waiting 30 seconds...")
            time.sleep(30)

    text2embedding.update(result)
    time.sleep(30)

# text2embedding = get_local_embeddings(texts + val_texts, model=EMBEDDER, batch_size=128, cache_name=CACHE_NAME)

embeddings = np.stack([text2embedding[text] for text in texts])
train_embeddings = np.stack([text2embedding[text] for text in texts])
val_embeddings = np.stack([text2embedding[text] for text in val_texts])

Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk 0:   0%|          | 0/32 [00:00<?, ?it/s]

API error: Error code: 429 - {'error': {'message': 'Rate limit reached for text-embedding-3-small in organization org-iMpg2C8uQw8mp4v2NZfryiVV on tokens per min (TPM): Limit 1000000, Used 985502, Requested 33541. Please try again in 1.142s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}; retrying in 3.0s... (2/3)
Saved 482 embeddings to /Users/jiaoweigong/Desktop/HypotheSAEs/emb_cache/jobposting_text-embedding-3-small/chunk_000.npy


Loading embedding chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Loaded 482 embeddings in 0.1s


...

Loading embedding chunks:   0%|          | 0/30 [00:00<?, ?it/s]

Loaded 13806 embeddings in 4.4s


Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk 30:   0%|          | 0/14 [00:00<?, ?it/s]

Saved 173 embeddings to /Users/jiaoweigong/Desktop/HypotheSAEs/emb_cache/jobposting_text-embedding-3-small/chunk_030.npy


**Train SAE** 

We will train a Matryoshka SAE with $M=256$, $k=8$, and $\text{prefix\_lengths} = [32, 256]$.  

With the Matryoshka loss, the SAE will learn to reconstruct the input from (1) just the first 32 neurons, and (2) all 256 neurons.  
This will produce 32 coarse-grained features, and 224 finer-grained features.  

See the README for more details about selecting SAE hyperparameters. 

In [37]:
checkpoint_dir = os.path.join(prefix, "checkpoints", CACHE_NAME)
sae = train_sae(embeddings=train_embeddings, val_embeddings=val_embeddings,
                M=256, K=8, matryoshka_prefix_lengths=[32, 256], 
                checkpoint_dir=checkpoint_dir)

  0%|          | 0/100 [00:00<?, ?it/s]

Early stopping triggered after 48 epochs
Saved model to ../checkpoints/jobposting_text-embedding-3-small/SAE_matryoshka_M=256_K=8_prefixes=32-256.pt


**Interpret neurons**  

Interpret a random subset of neurons in the SAE to sanity-check that the learned features, and their interpretations, seem reasonable. We generate and print labels for `n_random_neurons` neurons, and we also print out the top-activating texts for each neuron.

In [ ]:
# This instruction will be included in the neuron interpretation prompt.
# The below instructions are specific to my task.
# If you don't pass in task-specific instructions, there is a generic instruction (see src/interpret_neurons.py);
# task-specific instructions are optional, but they help produce hypotheses at the desired level of specificity.

TASK_SPECIFIC_INSTRUCTIONS = """All of the texts are job postings from firms.
Features should describe a specific aspect of the job posting language, such as job requirements, responsibilities, workplace culture, employee expectations, skills, benefits, career development, flexibility, diversity, or organizational priorities.

Avoid describing features only in terms of specific company names, brand names, or job titles. Prefer generalizable patterns in recruiting language.

For example:
- "emphasizes teamwork and collaboration"
- "stresses performance targets and measurable results"
- "highlights opportunities for promotion and career development"
- "emphasizes flexible schedules and work arrangements"
- "mentions diversity, inclusion, and belonging"
"""

results = interpret_sae(
    texts=texts,
    embeddings=train_embeddings,
    sae=sae,
    n_random_neurons=5,
    print_examples_n=3,
    task_specific_instructions=TASK_SPECIFIC_INSTRUCTIONS,
    interpreter_model=INTERPRETER_MODEL,
    interpret_llm_kwargs=INTERPRET_LLM_KWARGS,
)


Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Activations shape: (14274, 256)


Generating interpretations:   0%|          | 0/5 [00:00<?, ?it/s]


Neuron 140 (6.1% active): explicitly references Colorado-specific job details (e.g., CO locations and/or the Colorado compensation-range disclosure language)

Top activating examples:
1. For individuals assigned and/or hired to work in Colorado, is required by law to include a reasonable estimate of the compensation for this role. This compensation range is specific to the State of Colorado and takes into account various factor Manager, Business, Development, Account, Sales, Sales Manager, Manufacturing
2. Are you ready to grow your dream career while making others' vacation dreams come true? is a world premier organization for Vacation Ownership with resorts at destinations around the globe. Join our team and help deliver unforgettable experiences that make vacation dreams come true. Hospitality Job Fair – Sheraton Mountain Vista Villas & Marriott's StreamSide All candidates are guaranteed an interview! Openings at locations in Avon and Vail, CO! $1000 Starting Bonus! Full-Time, Part

**Generate hypotheses**

Generate hypotheses which are predictive of the target variable.

The `selection_method` parameter defines how we compute neuron predictiveness (see `src/select_neurons.py` for more details):
- "separation_score": E[target | top-activating examples] - E[target | zero-activating examples]
- "correlation": pearson(neuron activations, target variable)
- "lasso": select N nonzero features with an L1 regularized model

This cell outputs a dataframe with the following columns:
- `neuron_idx`: The index of the neuron in the SAE (if you're using multiple SAEs, this will be a global index across all of them).
- `source_sae`: The SAE that the neuron was selected from.
- `target_{selection_method}`: The predictiveness of the neuron for the target variable, using the selected `selection_method`.
- `interpretation`: The natural language interpretation of the neuron.
- `interp_fidelity_score`: The F1 fidelity score for how well the neuron's interpretation actually corresponds to its activation pattern.

In [39]:
selection_method = "correlation"

results = generate_hypotheses(
    texts=texts,
    labels=labels,
    embeddings=train_embeddings,
    sae=sae,
    cache_name=CACHE_NAME,
    selection_method=selection_method,
    n_selected_neurons=20,
    n_candidate_interpretations=1,
    task_specific_instructions=TASK_SPECIFIC_INSTRUCTIONS,
    interpreter_model=INTERPRETER_MODEL,
    annotator_model=ANNOTATOR_MODEL,
    interpret_llm_kwargs=INTERPRET_LLM_KWARGS,
    annotation_llm_kwargs=ANNOTATION_LLM_KWARGS,
    n_workers_annotation=N_WORKERS_ANNOTATION,
)

print("\nMost predictive features of ROA decline:")

pd.set_option("display.max_colwidth", None)
display(results.sort_values(by=f"target_{selection_method}", ascending=False).round(3))
pd.reset_option("display.max_colwidth")

Embeddings shape: (14274, 1536)


Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Activations shape: (14274, 256)

Step 1: Selecting top 20 predictive neurons

Step 2: Interpreting selected neurons


/Users/jiaoweigong/Desktop/HypotheSAEs/hypothesaes/select_neurons.py:130: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearsonr(activations[:, i], target)[0]


Generating interpretations:   0%|          | 0/20 [00:00<?, ?it/s]


Step 3: Scoring Interpretations
Found 0 cached items; annotating 2000 uncached items


Scoring neuron interpretation fidelity (20 neurons; 1 candidate interps per neuron; 100 examples to score each…


Most predictive features of ROA decline:


,neuron_idx,target_correlation,interpretation,f1_fidelity_score
0,10,0.527,"describes front-office financial markets risk/trading technology or regulatory/compliance initiatives using domain-specific jargon (e.g., Market Risk, FRTB/IMA, VaR, KYC-AML, desk/trader workflows, margin/exposure) and end-to-end delivery artifacts like BRDs, UAT, model validation, and data quality controls",0.675
1,15,0.435,"describes corporate finance/investment banking work involving capital markets transactions and quantitative financial modeling/valuation (e.g., M&A, IPOs, debt/equity raises, swaps/derivatives, risk models, Bloomberg/FactSet) in a global financial services context",0.507
2,19,0.379,"describes retail branch banking sales behaviors like proactive outbound calling, appointment setting, lobby engagement, and using a defined sales process to deepen customer relationships and grow share of wallet",1.000
4,1,0.295,"describes a retail banking relationship-building role focused on advising clients on personal financial goals and educating them on digital banking solutions, with explicit mention of structured onboarding/training and clear internal advancement paths (e.g., Relationship Manager/Financial Advisor/branch management)",1.000
5,29,0.291,"lists extensive enterprise software architecture/engineering technical requirements (e.g., .NET/Java, cloud platforms like Azure/AWS, APIs/web services, CI/CD tools, databases) in a long skills/technology stack section",0.611
7,9,0.254,"focuses on mortgage/loan closing documentation administration (preparing/reviewing loan documents, coordinating closings, ensuring lien perfection and regulatory compliance, handling funding/booking and post-closing collateral files)",0.734
8,22,0.246,"industrial mechanical service/repair work on pumps and related equipment (e.g., dismantling/assembling pumps, welding/pipe-fitting, troubleshooting pump performance at customer sites)",0.649
10,8,0.220,"includes a standardized job header listing logistics/compliance fields like primary location, relocation offered, employment status, travel percentage, and non-compete status",0.810
15,40,0.180,"includes a standardized corporate intro slogan and branding block (e.g., 'At [company], we've got a place for you!' / 'Energize your career') followed by an all-caps 'PRIMARY PURPOSE OF POSITION' section heading",0.925
18,66,0.176,"mentions mechanical seal/turbomachinery order engineering work, including creating CAD drawings/models in SolidWorks or AutoCAD for seal components and critical dimensions",0.413


**Evaluate held-out generalization**

Finally, we evaluate whether these are good hypotheses by testing whether their natural language interpretations can predict the target variable.  

We compute annotations for each hypothesized concept on a holdout set (not seen during SAE training & feature selection).

After annotation, we output a dataframe with the following columns:
- `hypothesis`: The natural language hypothesis (which came from interpreting a predictive neuron in the SAE)
- `separation_score`: How much the target variable differs when the concept is present vs. absent (i.e., $E[Y\mid\text{concept} = 1] - E[Y\mid\text{concept} = 0]$).
- `separation_pvalue`: The t-test p-value of the null hypothesis that the separation score is 0 (i.e., the concept is not associated with the target variable).
- `regression_coef`: The coefficient of the concept in a multivariate linear regression of the target variable on all concepts.
- `regression_pval`: The p-value of the null hypothesis that the regression coefficient is 0.
- `feature_prevalence`: The fraction of examples that contain the concept.

Additionally, we output the evaluation metrics used in the paper:
- Significant hypotheses: the number of hypotheses that are significant in the multivariate regression at a specified significance level (default $0.1$) after Bonferroni correction. You can pass in a different significance level using the `corrected_pval_threshold` parameter.
- AUC or $R^2$: how well the hypotheses collectively predict the target variable in the multivariate regression.


In [41]:
results_3 = (
    results.assign(abs_corr=results["target_correlation"].abs())
           .sort_values("abs_corr", ascending=False)
           .head(3)
           .drop(columns="abs_corr")
)

holdout_df = pd.read_json(os.path.join(base_dir, "JobPosting_holdout.json"), lines=True)
holdout_texts = holdout_df["text_clean"].tolist()
holdout_labels = holdout_df["decline"].astype(int).values

metrics, evaluation_df = evaluate_hypotheses(
    hypotheses_df=results_3,
    texts=holdout_texts,
    labels=holdout_labels,
    cache_name=CACHE_NAME,
    annotator_model=ANNOTATOR_MODEL,
    annotation_llm_kwargs=ANNOTATION_LLM_KWARGS,
    n_workers_annotation=N_WORKERS_ANNOTATION, # Please lower this parameter if you are running into API rate limits
)

pd.set_option("display.max_colwidth", None)
display(evaluation_df.sort_values(by="separation_score", ascending=False).round(3))
pd.reset_option("display.max_colwidth")

print("\nHoldout Set Metrics:")
print(metrics)
print(f"Significant hypotheses: {metrics['Significant'][0]}/{metrics['Significant'][1]} "
      f"(p < {metrics['Significant'][2]:.3e})")

Step 1: Annotating texts with 3 hypotheses
Found 735 cached items; annotating 3078 uncached items


Annotating:   0%|          | 0/3078 [00:00<?, ?it/s]

Step 2: Computing predictiveness of hypothesis annotations
         Current function value: 0.299890
         Iterations: 35
Error fitting model: Singular matrix, trying OLS instead


/Users/jiaoweigong/Desktop/HypotheSAEs/hypothesaes/evaluation.py:150: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  results = model.fit()
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3862: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/jiaoweigong/Desktop/HypotheSAEs/hypothesaes/evaluation.py:118: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  _, p_value = ttest_ind(pos_vals, neg_vals)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:601: RuntimeWarning:

,hypothesis,separation_score,separation_pval,regression_coef,regression_pval,feature_prevalence
1,"describes corporate finance/investment banking work involving capital markets transactions and quantitative financial modeling/valuation (e.g., M&A, IPOs, debt/equity raises, swaps/derivatives, risk models, Bloomberg/FactSet) in a global financial services context",0.097,0.006,0.096,0.006,0.056
2,"describes retail branch banking sales behaviors like proactive outbound calling, appointment setting, lobby engagement, and using a defined sales process to deepen customer relationships and grow share of wallet",-0.052,0.635,-0.046,0.670,0.006
0,"describes front-office financial markets risk/trading technology or regulatory/compliance initiatives using domain-specific jargon (e.g., Market Risk, FRTB/IMA, VaR, KYC-AML, desk/trader workflows, margin/exposure) and end-to-end delivery artifacts like BRDs, UAT, model validation, and data quality controls",NaN,NaN,0.000,0.590,0.000



Holdout Set Metrics:
{'auroc': 0.5321839080459769, 'auprc': 0.9146102753590628, 'Significant': (1, 3, 0.03333333333333333)}
Significant hypotheses: 1/3 (p < 3.333e-02)


API error: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}; retrying in 2.0s... (2/3)
API error: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}; retrying in 2.0s... (2/3)
API error: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}; retrying in 2.0s... (2/3)
API error: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using th

## Cosine-Similarity Benchmark

As a follow-up suggested in the assignment 3.b., I considered using cosine similarity as a cheaper alternative to LLM-based holdout annotation for the same three hypotheses. The code below embeds each hypothesis, compares it with the holdout-text embeddings, and examines whether the similarity scores are associated with the ROA-decline label. A fuller benchmark would also compare these cosine-similarity scores directly with the posting-level LLM annotations to assess whether the two approaches produce similar conclusions. Because I started this extension late and the notebook kernel had restarted, I did not rerun the full pipeline and leave the code below as a possible follow-up.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pointbiserialr

hypotheses = results_3["hypothesis"].tolist()
cosine_texts = holdout_texts + hypotheses

cosine_dict = get_openai_embeddings(cosine_texts, model=EMBEDDER, batch_size=16, n_workers=1, cache_name=CACHE_NAME)

holdout_emb = np.stack([cosine_dict[x] for x in holdout_texts])
hypothesis_emb = np.stack([cosine_dict[x] for x in hypotheses])
similarities = cosine_similarity(holdout_emb, hypothesis_emb)

cosine_results = []
for j, h in enumerate(hypotheses):
    r, p = pointbiserialr(holdout_labels, similarities[:, j])
    cosine_results.append({"hypothesis": h, "correlation": r, "p_value": p,
                           "mean_similarity": similarities[:, j].mean()})

cosine_results = pd.DataFrame(cosine_results)
display(cosine_results.round(3))